<a href="https://colab.research.google.com/github/gsaini26/Data-Files/blob/main/AI_Reminders_to_Prevent_No_Shows.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
from google import genai
from google.genai import types
from datetime import datetime
from google.colab import userdata
import time # Import the time module

# --- Control Flag for Mock API ---
USE_MOCK_API = False # Set to False to use actual Gemini API, True to use mock responses
# ---------------------------------

# 1. SETUP: Connect to Gemini API
# Replace 'YOUR_API_KEY' with your actual Google AI Studio API key

GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GOOGLE_API_KEY)
# Removed 'model = "gemini-2.5-flash"' from here, as the model is specified in the generate_content call


# 2. DATA LOADING: Read the uploaded file
df = pd.read_csv('https://raw.githubusercontent.com/gsaini26/Data-Files/refs/heads/main/clinical_optimizer_showcase.csv')

# --- DIAGNOSTIC: Print columns to identify correct 'NoShow' column ---
# print("DataFrame columns:", df.columns)
# -------------------------------------------------------------------

# 3. LOGIC GATING: Identify high-risk patients (Wait time > 14 days or Friday)
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay'])
df['is_friday'] = df['AppointmentDay'].dt.day_name() == 'Friday'

# Apply the function to the entire DataFrame to generate SMS for all patients
# The filtering logic for nudges is handled within the generate_optimized_sms function
# Renamed df to patients_to_process_df for clarity when applying the function
patients_to_process_df = df.copy()

def generate_optimized_sms(row):
    """Function to call Gemini for a personalized nudge based on risk factors."""

    # System Instruction for the AI
    system_prompt = (
        "You are an AI Operations Agent. Your task is to write a 160-character SMS reminder "
        "using a psychological nudge. "
    )

    # Decision Matrix for nudge logic
    if row['is_friday']:
        nudge_type = "Loss Aversion (Highlight the scarcity and high demand for end-of-week slots)."
    elif row['Date.diff'] > 21:
        nudge_type = "Reciprocity (Mention the doctor has specifically prepared their medical chart/file)."
    elif row['Age'] < 35 and row['Date.diff'] > 10:
        nudge_type = "Implementation Intention (Ask if they have planned their travel/route to the clinic)."
    elif row['Scholarship'] == True: # Assuming 'Scholarship' column contains boolean or equivalent values
        nudge_type = "Social Proof (Reinforce their identity as a consistent, health-conscious program member)."
    else: # Default for low-risk patients
        nudge_type = "Standard (Standard professional reminder focusing on clarity and the upcoming date/time)."

    user_prompt = f"Patient ID: {row['PatientId']}. Appointment: {row['AppointmentDay'].date()}. Nudge type: {nudge_type}"

    # --- Conditional API Call or Mock Response ---
    if USE_MOCK_API:
        # Return a mock response to avoid API calls during debugging
        return f"[MOCK SMS for Patient {row['PatientId']} with nudge: {nudge_type.split('(')[0].strip()}]"
    else:
        try:
            time.sleep(12) # Increased delay to 12 seconds to prevent rate limiting (5 calls/min limit)
            # Correctly using client.models.generate_content with model specified as an argument
            response = client.models.generate_content(model='gemini-2.5-flash', contents=system_prompt + user_prompt)
            return response.text.strip()
        except Exception as e:
            print(f"Error generating SMS: {e}") # Print the actual error message
            return "Standard reminder: We look forward to seeing you tomorrow."

# 4. EXECUTION: Generate the messages
print("Generating optimized nudges for all patients...")
patients_to_process_df['Optimized_SMS'] = patients_to_process_df.apply(generate_optimized_sms, axis=1)

# 5. OUTPUT: Create the "Revenue Recovery" report
recovery_list = patients_to_process_df[['PatientId', 'AppointmentDay', 'Date.diff', 'Optimized_SMS']]
recovery_list.to_csv('revenue_recovery_list.csv', index=False)

print("Process complete. File saved: 'revenue_recovery_list.csv'")

Generating optimized nudges for all patients...
Process complete. File saved: 'revenue_recovery_list.csv'
